In [ ]:
import requests
import requests_cache
from heuristic_clustering import HeuristicClustering
from get_address_info import GetAddressInfo
import datetime
import pymysql
import pandas as pd
import os

In [2]:
data = pd.read_csv("test.csv")
data.head()

,id,address,location,btc_format,num_txs,last_txs,entity,timezone,offsset,arkham_entity,arkham_label,order_id
0,367661,14mp3x1GVCrZgRqbDCAyUJdFjru7K4sZoc,AZ,Valid Format,7883,2016-03-29,Entity 0,Asia/Baku,4.0,NaN,NaN,0
1,2442712,1NekBxxvVPjMDYZ2DQYHuWffW51pfQGDq7,China,Valid Format,7541,2024-12-17,Entity 1,Asia/Shanghai,8.0,NaN,NaN,1
2,639470,1ART8MzPqABmqWz4SsaKLZfrwb9HjNF5dv,Netherlands,Valid Format,6755,2016-05-02,Entity 2,Europe/Amsterdam,1.0,NaN,NaN,2
3,110907,1B9x2xeyzU7X2m6oqhftFPMmhjQdZvFZt1,taiwan,Valid Format,6081,2013-08-29,Entity 3,Asia/Taipei,8.0,NaN,Miner,3
4,154469,19SrA28QN8Eopd7oEWAtmy7dTm2CG7Wuck,PAKISTAN,Valid Format,5349,2024-03-23,Entity 4,Asia/Karachi,5.0,NaN,NaN,4


In [ ]:
def data_write(chunk_size, batch_blockchain_data, batch_tx_inputs, batch_tx_outputs, batch_entities_heur):
    conn = pymysql.connect(
    host = '127.0.0.1',
    user = 'root',
    password =  os.getenv("PASSWORD_MY_SQL"),
    database = 'data'
    )
    while batch_blockchain_data:
        with conn.cursor() as cursor:
            chunk = batch_blockchain_data[:chunk_size]
            sql = "INSERT IGNORE INTO blockchain_data (txid, num_inputs, num_outputs, fee, mempool_entry_time, block_height) VALUES (%s, %s, %s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_blockchain_data = batch_blockchain_data[chunk_size:]
            conn.commit()

        
    while batch_tx_inputs:
        with conn.cursor() as cursor:
            chunk = batch_tx_inputs[:chunk_size]
            sql = "INSERT IGNORE INTO tx_inputs (txid, input_order, address, value) VALUES (%s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_tx_inputs = batch_tx_inputs[chunk_size:]
            conn.commit()

    while batch_tx_outputs:
        with conn.cursor() as cursor:
            chunk = batch_tx_outputs[:chunk_size]
            sql = "INSERT IGNORE INTO tx_outputs (txid, output_order, address, value) VALUES (%s, %s, %s, %s)"
            cursor.executemany(sql, chunk)
            batch_tx_outputs = batch_tx_outputs[chunk_size:]
            conn.commit()
    
    while batch_entities_heur:
        with conn.cursor() as cursor:
            chunk = batch_entities_heur[:chunk_size]
            sql = "INSERT IGNORE INTO entities_heur (entity, address) VALUES (%s, %s)"
            cursor.executemany(sql, chunk)
            batch_entities_heur = batch_entities_heur[chunk_size:]
            conn.commit()

In [4]:
path = r"/home/matej/btc_timezone_analysis/data_collection/Webshare_100_proxies.txt"
proxies_dict_list = []

# Open the text file containing the proxies in read mode with UTF-8 encoding
with open(path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        
        # split the TXT format IP:PORT:USER:PASS
        ip, port, user, pwd = line.split(":")
        
        # create a dict directly for requests
        proxy_dict = {
            "http": f"http://{user}:{pwd}@{ip}:{port}",
            "https": f"http://{user}:{pwd}@{ip}:{port}"
        }

        proxies_dict_list.append(proxy_dict)
        

In [5]:
session = requests_cache.CachedSession('api_cache', expire_after=datetime.timedelta(days=30))

In [17]:
list_to_do = [4211, 4217, 4378, 4472, 4498, 4652, 4730, 4996, 5013, 5080]

In [18]:
a = 50
for index, row in data.iterrows():
    if index in list_to_do:
        print(index)
        address = row["address"]
        entity = row["entity"]
        heur = HeuristicClustering(address, session, a, proxies_dict_list)
        heur.heuristic_clus()
        a = heur.a
        addresses_heur = [[entity,addr] for addr in heur.old_addresses]

        data_write(1000, heur.blockchain_final_data, heur.inputs_final_data, heur.outputs_final_data, addresses_heur)
        print(f"Address {address} written with {len(heur.old_addresses)}")
    



4211
['17EFZ829NBT2WETLj3wJ5YUfXVaGckuUgs']
['3AxstDkandd3NyoxGAKQFNZpSXKfnmuj43', '35GRrMYqi5kLDWGbV4cPHzPmURo3eEB3Wn', '3Hg7j7K5zcJGoCK8gTCPbk5W4mgFHncpaP', '3C2p1E4bgBr7vwd2r4H5KMAG6xqHeTjmmt', '3MW39ZWje7tYDuee9ZTr9rTPHuHcngf1Sx', '31niKWY7j6Fw5f9iPvMcExX4f5hmdGX3P3', '34SNsDbr4VayiihdLAFvNvibNp7oweV4aA', '3MNfxBu3W1v9cugGcZoRACjwaVzFY8LgUL', '3ALaap91V5Yi5BvNMrD8Azrd9oGRtSXDhq', '37TmnP5TfdSMyMXWGZhJBUr1mDwD4BwR9T', '3AaVo4921Qma4FGJH5CXkYbFZMfgJyE9AJ', '32J794ysXL3TZdcduWbjy1PQ9kb5ni2LNk', '3GTxWfpLjkjwtcP1rYKPeDKAtw3CKbqPck', '3Qmn2MCtrrXwezEHo4wKUm6N2TPVfhSPtw', '336iZZgeQciFVXMRn5qXNCUQ8kHTXe84R6', '3B9N7f6CBwxrRSFSmhcm56at6pt5qvh3kw', '3DSTWHWMpMiyYeBX7ueWus7pVv9CSThHPp', '3MUgVLpJLGnusHuC9D3956h19dviSabBQq', '3GH94yARPsqExBu5VMA9LkMaezfvasCK1z', '3FoiKfyakp929Xpoz2qJXtysUKtieg9uBq', '35dKPAExqHJpUWPmCBgGRYjoAQmehfyezS', '376JZtDnqm8FAux6RaNWC4ENVoLPRSj2uT', '3LE2FNKNHRLBRjbq3ZHStAxveav3qnGfrC', '39XFcBCmFFdDuQq8k55UUnmR8ZQdGZMXpG', '3EZUdJrdekZDk4aSM81rTRvka9Z1aQHdG7', '38Vf

: 

In [7]:
address = "1Evg5EEKjbqS7bNKnz6kKi4wY3UmDBApFn"
a = 0
address_analysis = GetAddressInfo(address, session, a, proxies_dict_list)
address_analysis.fetch_and_extract()
#heur = HeuristicClustering(address, session, a, proxies_dict_list)
#heur.heuristic_clus()

0.1431427001953125


In [6]:
len(address_analysis.batch_blockchain_data)

804

In [7]:
address_analysis.batch_blockchain_data

[['9d1cb741947cc2e1e512e395e865e266ed394b0a8db654d3717d362f45025f2a',
  29,
  1,
  54.11506953705468,
  '2017-06-09 06:32:45',
  470477],
 ['648d6068b6d18e52323990f34d3b4ffb6243f4098e4e776a0617460ce97adf16',
  1,
  2,
  20.164444444444445,
  '2015-12-07 14:55:46',
  387168],
 ['68305694b87d5175d924b811b6f3d3d0d709da3a4e5d967b6ab68a083c6ca38a',
  2,
  2,
  20.12064343163539,
  '2015-12-06 15:20:01',
  387021],
 ['3bef9c743d158ff2a0867faf89a2bd3a69afbcf955eeceabc5d48d529dc901ed',
  2,
  2,
  20.072386058981234,
  '2015-12-02 15:55:05',
  386396],
 ['a04a87f0c8f4272a582441efd9a39747fec6f181f3c4efa0ae7ec7ea04b6dfbe',
  2,
  2,
  20.0455764075067,
  '2015-11-30 14:45:39',
  386050],
 ['db3f7021a37002c4415a930ecbfa5b83633b22e4851c511cfff0eb7a919e5cd1',
  1,
  2,
  32.94690265486726,
  '2015-11-26 15:06:18',
  385452],
 ['531fd0ffa9d1f525e5319b5be27f7d327d7281f8d959d6f4b6a921cedb0fc4ab',
  2,
  2,
  18.74331550802139,
  '2015-11-25 15:07:47',
  385293],
 ['eb0dfd2b893c2cc84f22adeea1f6971248df

In [10]:
address = "bc1pzuda4vsgaqqvnyqka05mpaw8jewlnmj3g95fp4y7kh98s5pjzydqme8dwd"
url = f"https://blockchain.info/rawaddr/{address}"
r = requests.get(url)
r = r.json()
r



{'hash160': None,
 'address': 'bc1pzuda4vsgaqqvnyqka05mpaw8jewlnmj3g95fp4y7kh98s5pjzydqme8dwd',
 'n_tx': 4,
 'n_unredeemed': 1,
 'total_received': 27063,
 'total_sent': 17063,
 'final_balance': 10000,
 'txs': [{'hash': 'c1a69100367d47ee9db89d935a49510459132363cb5c63471777cd44682d4561',
   'ver': 2,
   'vin_sz': 1,
   'vout_sz': 3,
   'size': 265,
   'weight': 733,
   'fee': 520,
   'relayed_by': '0.0.0.0',
   'lock_time': 961196,
   'tx_index': 3422391758264750,
   'double_spend': False,
   'time': 1785957393,
   'block_index': 961197,
   'block_height': 961197,
   'inputs': [{'sequence': 4294967293,
     'witness': '024730440220399538385cb942096d69f0317a754c803aa095f9f913c07771ed73851904b16b02200efcc98c22d88310529a19352c0778323430d5b4e03ef3842dfafca54590473c012103cad95529ce8b6b25cf76f39d882d5a4c6446aa51ffa58226fa27ab77f0b07e23',
     'script': '',
     'index': 0,
     'prev_out': {'type': 0,
      'spent': True,
      'value': 22720,
      'spending_outpoints': [{'tx_index': 34223917

In [44]:
#Blockchain data creation
data = pd.DataFrame(r["txs"])

blockchain_data = pd.DataFrame()
blockchain_data["txid"] = data["hash"]
blockchain_data["num_inputs"] = data["vin_sz"]
blockchain_data["num_outputs"] = data["vout_sz"]
blockchain_data["fee"] = data["fee"]/data["size"]
blockchain_data["mempool_entry_time"] = pd.to_datetime(
        data['time'], 
        unit='s',       
        utc=True        
    ).dt.strftime("%Y-%m-%d %H:%M:%S")
blockchain_data["block_height"] = data["block_height"]

In [43]:
#Tx inputs table data creation
df_inputs = data[['hash', 'inputs']].explode('inputs').dropna(subset=['inputs'])
df_inputs = df_inputs.reset_index(drop=True)
inputs_expanded = pd.DataFrame(df_inputs['inputs'].tolist())
inputs_df = df_inputs[['hash']].join(inputs_expanded).reset_index(drop=True)
inputs_df["prev_out"] = inputs_df['prev_out'].to_dict()
df = pd.DataFrame(inputs_df["prev_out"].tolist(), index = inputs_df.index)
final_inputs = pd.concat([inputs_df, df], axis = 1)

tx_inputs = pd.DataFrame()
tx_inputs["txid"] = final_inputs["hash"]
tx_inputs["input_order"] = final_inputs["index"]
tx_inputs["address"] = final_inputs["addr"]
tx_inputs["value"] = final_inputs["value"]/100000000

In [48]:
#Tx outputs table data creation
df_outputs = data[['hash', 'out']].explode('out').dropna(subset=['out'])
df_outputs = df_outputs.reset_index(drop=True)
outputs_expanded = pd.DataFrame(df_outputs['out'].tolist())
outputs_df = df_outputs[['hash']].join(outputs_expanded).reset_index(drop=True)

tx_outputs = pd.DataFrame()
tx_outputs["txid"] = outputs_df["hash"]
tx_outputs["output_order"] = outputs_df["n"]
tx_outputs["address"] = outputs_df["addr"]
tx_outputs["value"] = outputs_df["value"]/100000000
tx_outputs

,txid,output_order,address,value
0,c1a69100367d47ee9db89d935a49510459132363cb5c63...,0,bc1q7jvscxyxrnqr393glx74u6g25uwhqc56aakalf,0.000022
1,c1a69100367d47ee9db89d935a49510459132363cb5c63...,1,bc1q7fazrw5j2gz3pkpd5mtzj562vugt98z6h20xn7,0.000100
2,c1a69100367d47ee9db89d935a49510459132363cb5c63...,2,bc1pzuda4vsgaqqvnyqka05mpaw8jewlnmj3g95fp4y7kh...,0.000100
3,0530871080f2c50146136e6c4ed2880b3368fec585363d...,0,bc1qhd35usefn8whvtg49cvzjrns7jc3xh55588zj0,0.000227
4,af72ec1bff288d14cd3d476cee8d352face9a66a79adc8...,0,bc1pzuda4vsgaqqvnyqka05mpaw8jewlnmj3g95fp4y7kh...,0.000120
5,8a02bb1faaaade3df1f48c9e67ee736ee2f17ba209e44b...,0,bc1pzuda4vsgaqqvnyqka05mpaw8jewlnmj3g95fp4y7kh...,0.000051
